## Decorators

A decorator is a wrapper around a function. It lets you add extra behaviour to a function without changing the function itself.

**Example:** `@app.route` does the same thing — it wraps your function with Flask behaviour that says "when this URL is hit, call this function.

This is why decorators are powerful in backend development. You can add behaviour to many functions without repeating code. For example a real Atlassian codebase might have:

In [ ]:
@app.route("/users", methods=["GET"])
@require_auth          # checks if user is logged in
@rate_limit(100)       # max 100 requests per minute
def get_users():
    ...

Three decorators stacked — each one wrapping the function with extra behaviour.

## Real production example: 

You have 10 endpoints and every single one needs to check if the user is logged in before doing anything:

In [ ]:
@app.route("/users", methods=["GET"])
def get_users():
    token = request.headers.get("Authorization")
    if not token:
        return jsonify({"error": "Not logged in"}), 401
    # actual logic here

@app.route("/tasks", methods=["GET"])
def get_tasks():
    token = request.headers.get("Authorization")
    if not token:
        return jsonify({"error": "Not logged in"}), 401
    # actual logic here

@app.route("/projects", methods=["GET"])
def get_projects():
    token = request.headers.get("Authorization")
    if not token:
        return jsonify({"error": "Not logged in"}), 401
    # actual logic here

You're copying the same auth check into every single function. That's messy and if you need to change the logic you have to update 10 places.

### With a decorator:

In [ ]:
def require_auth(func):
    def wrapper(*args, **kwargs):
        token = request.headers.get("Authorization")
        if not token:
            return jsonify({"error": "Not logged in"}), 401
        return func(*args, **kwargs)
    return wrapper

@app.route("/users", methods=["GET"])
@require_auth
def get_users():
    # actual logic here

@app.route("/tasks", methods=["GET"])
@require_auth
def get_tasks():
    # actual logic here

Auth logic written once, applied everywhere. Change it in one place, updates everywhere.

Full code:

In [ ]:
from functools import wraps
from flask import jsonify, request, app

def require_auth(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        token = request.headers.get("Authorization")
        if not token:
            return jsonify({"error": "Not authorised"}), 401
        return func(*args, **kwargs)
    return wrapper

In [ ]:
@app.route("/users/<email>", methods=["GET"])
@require_auth
def get_user(email):
    user = user_db.get(email)
    if user is None:
        return jsonify({"error": "User not found"}), 404
    return jsonify({"name": user.name, "email": user.email}), 200